# TFM · GNN × GDELT × SP500

Orquestador de entrenamiento. Este notebook ejecuta el pipeline de extremo a extremo:

1. Clonar el repo y instalar dependencias.
2. Descargar datos de GDELT y financieros.
3. Preprocesar y construir el dataset.
4. Entrenar la HGNN con walk-forward.
5. Evaluar y comparar con baselines.

Editar los parámetros de `config.py` (en el repo) para ajustar el experimento. No conviene poner lógica nueva en este notebook: si algo se reusa, va a `src/`.

## 1. Setup

En la primera ejecución en Colab hay que clonar el repo e instalar PyG. PyTorch Geometric requiere instalación específica según la versión de torch+CUDA de Colab.

In [ ]:
# Detectar si estamos en Colab
try:
    import google.colab  # type: ignore
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
print('En Colab:', EN_COLAB)

In [ ]:
if EN_COLAB:
    # AJUSTAR: reemplazar <usuario> por vuestro usuario de GitHub
    # Si el repo es privado, guardar el token en los secrets de Colab y descomentar las dos líneas siguientes:
    # from google.colab import userdata
    # token = userdata.get('GITHUB_TOKEN')
    # !git clone https://{token}@github.com/<usuario>/ai-gnn-gdelt-sp500.git
    !git clone https://github.com/<usuario>/ai-gnn-gdelt-sp500.git
    %cd ai-gnn-gdelt-sp500
    !pip install -q -r requirements.txt

In [ ]:
import sys
from pathlib import Path
# Asegurar que el cwd está en el repo (importar src.* requiere esto)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

## 2. Descarga de datos

Definir el rango de fechas y descargar GDELT + SP500. Ajustar en `config.py` o sobreescribir aquí.

In [ ]:
from datetime import date
import config
from src.datos.descarga_gdelt import descargar_rango
from src.datos.descarga_financiero import descargar_sp500, descargar_vix, descargar_macro_fred

# Para una primera prueba conviene usar un rango corto, luego ampliar
FECHA_INI = date(2024, 1, 1)
FECHA_FIN = date(2024, 6, 30)

rutas_gdelt = descargar_rango(FECHA_INI, FECHA_FIN)
print(f'GDELT: {len(rutas_gdelt)} días descargados')

In [ ]:
precios = descargar_sp500(FECHA_INI, FECHA_FIN)
vix = descargar_vix(FECHA_INI, FECHA_FIN)
try:
    macro = descargar_macro_fred(FECHA_INI, FECHA_FIN)
except Exception as e:
    print('Macro FRED no disponible:', e)
    macro = None

print('SP500:', len(precios), 'sesiones')
print('VIX:', len(vix), 'puntos')

## 3. Preprocesado y construcción del dataset

In [ ]:
from src.datos.preprocesar import preprocesar_rango, agregar_eventos_por_dia_y_par
from src.datos.etiquetas import construir_etiquetas_diarias
from src.datos.dataset import DatasetGrafoDiario

df_eventos = preprocesar_rango(rutas_gdelt)
df_agregado = agregar_eventos_por_dia_y_par(df_eventos)
print('Eventos preprocesados:', len(df_eventos))
print('Aristas agregadas:', len(df_agregado))

In [ ]:
etiquetas = construir_etiquetas_diarias(precios)
print(etiquetas.head())
print('Distribución de clases:')
print(etiquetas['clase'].value_counts().sort_index())

In [ ]:
# Filtrar etiquetas a fechas para las que tenemos datos de GDELT
import pandas as pd
fechas_gdelt = set(df_agregado['fecha'].dt.normalize().unique())
etiquetas_validas = etiquetas[etiquetas['fecha_grafo'].isin(fechas_gdelt)].reset_index(drop=True)
print(f'Etiquetas alineadas: {len(etiquetas_validas)}/{len(etiquetas)}')

dataset = DatasetGrafoDiario(
    eventos_agregados=df_agregado,
    etiquetas=etiquetas_validas,
    precios_sp500=precios,
    macro=macro,
    vix=vix,
    precomputar=False,
)
print('Dataset:', len(dataset), 'muestras')

## 4. Walk-forward + entrenamiento

In [ ]:
import torch
from src.entrenamiento.walkforward import construir_folds
from src.entrenamiento.loop import entrenar_walkforward

dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', dispositivo)

folds = construir_folds(
    fechas=etiquetas_validas['fecha_grafo'],
    num_folds=3,                # AJUSTAR: depende del rango temporal
    train_inicial_dias=60,      # AJUSTAR
    validacion_dias=20,         # AJUSTAR
)
print(f'{len(folds)} folds construidos')

In [ ]:
resultado = entrenar_walkforward(
    dataset=dataset,
    folds=folds,
    semillas=(0, 1, 2),
    dispositivo=dispositivo,
)
print(resultado.resumen_final())

## 5. Baselines

Comparación con modelos clásicos para aislar la aportación de la GNN.

In [ ]:
from src.modelo.baselines import (
    BaselineNaive, construir_matriz_features,
    entrenar_logistica, entrenar_xgboost,
)
from src.entrenamiento.evaluacion import calcular_metricas

# Construir X, y aplanados desde el dataset entero
X, y = construir_matriz_features(dataset)
print('X shape:', X.shape, '| y shape:', y.shape)

# Usar el primer fold como referencia rápida
f0 = folds[0]
X_tr, y_tr = X[f0.idx_train], y[f0.idx_train]
X_va, y_va = X[f0.idx_val], y[f0.idx_val]

naive = BaselineNaive().fit(X_tr, y_tr)
logr = entrenar_logistica(X_tr, y_tr)
xgb = entrenar_xgboost(X_tr, y_tr)

for nombre, modelo in [('naive', naive), ('logística', logr), ('xgboost', xgb)]:
    m = calcular_metricas(y_va, modelo.predict(X_va))
    print(f'{nombre:>10s}: {m.resumen()}')